In [ ]:
from dotenv import load_dotenv, find_dotenv

assert load_dotenv(find_dotenv(usecwd=False)), "The .env file was not loaded."

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from drn import DRN, drn_cutpoints, train
import seaborn as sns

torch.set_num_threads(1)

In [ ]:
DATA_DIR = Path("data/processed/synth")
x_train = pd.read_csv(DATA_DIR / "x_train.csv")
x_val = pd.read_csv(DATA_DIR / "x_val.csv")
x_test = pd.read_csv(DATA_DIR / "x_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv")

X_train = torch.Tensor(x_train.values)
X_val = torch.Tensor(x_val.values)
X_test = torch.Tensor(x_test.values)
Y_train = torch.Tensor(y_train.values).flatten()
Y_val = torch.Tensor(y_val.values).flatten()

In [ ]:
MODEL_DIR = Path("models/synth")
glm = torch.load(MODEL_DIR / "glm.pkl", weights_only=False)
ddr = torch.load(MODEL_DIR / "ddr.pkl", weights_only=False)
drn = torch.load(MODEL_DIR / "drn.pkl", weights_only=False)

In [ ]:
PLOT_DIR = Path("plots/synth")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Training

In [ ]:
train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
val_dataset = torch.utils.data.TensorDataset(X_val, Y_val)

# TODO: Load in the best parameters from the hyperparameter search pickle file
best_drn_params = [0.001, 0.0, 0.0, 0.0, 0.025, 10]

c_0 = min(Y_train.min().item() * 1.05, 0.0)
c_K = Y_train.max().item() * 1.05
cutpoints_DRN = drn_cutpoints(
    c_0,
    c_K,
    proportion=best_drn_params[-2],
    y=Y_train.detach().numpy(),
    min_obs=best_drn_params[-1],
)

torch.manual_seed(23)
drn_small_kl = DRN(
    glm=glm,
    cutpoints=cutpoints_DRN,
    hidden_size=256,
    num_hidden_layers=3,
    baseline_start=False,
    dropout_rate=0.1,
    kl_alpha=best_drn_params[1],
    mean_alpha=best_drn_params[2],
    dv_alpha=best_drn_params[3],
    kl_direction="forwards",
    loss_metric="jbce",
    learning_rate=best_drn_params[0],
)

train(
    model=drn_small_kl,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=256,
    epochs=2000,
    patience=50,
    print_details=True,
    log_interval=1,
)
drn_small_kl.eval()

In [ ]:
best_drn_params = [0.001, 0.05, 0.0, 0.0, 0.025, 10]

cutpoints_DRN = drn_cutpoints(
    c_0,
    c_K,
    proportion=best_drn_params[-2],
    y=Y_train.detach().numpy(),
    min_obs=best_drn_params[-1],
)

torch.manual_seed(23)
drn_large_kl = DRN(
    glm=glm,
    cutpoints=cutpoints_DRN,
    hidden_size=256,
    num_hidden_layers=3,
    baseline_start=False,
    dropout_rate=0.1,
    kl_alpha=best_drn_params[1],
    mean_alpha=best_drn_params[2],
    dv_alpha=best_drn_params[3],
    kl_direction="forwards",
    loss_metric="jbce",
    learning_rate=best_drn_params[0],
)

train(
    model=drn_large_kl,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=256,
    epochs=2000,
    patience=50,
    lr=best_drn_params[0],
    print_details=True,
    log_interval=1,
)
drn_large_kl.eval()

# Visualisation

In [ ]:
GRID_SIZE = 3000
grid = torch.linspace(0.0, np.max(y_train) * 1.1, GRID_SIZE).unsqueeze(-1)[1:]
logprob_drn = drn.distributions(X_test).log_prob(grid).detach().cpu().numpy().T
logprob_drn_small_kl = (
    drn_small_kl.distributions(X_test).log_prob(grid).detach().cpu().numpy().T
)
logprob_drn_large_kl = (
    drn_large_kl.distributions(X_test).log_prob(grid).detach().cpu().numpy().T
)
logprob_ddr = ddr.distributions(X_test).log_prob(grid).detach().cpu().numpy().T
logprob_baseline = glm.distributions(X_test).log_prob(grid).detach().cpu().numpy().T

In [ ]:
# Convert log-probs to probabilities
prob_drn = np.exp(logprob_drn)
prob_drn_small_kl = np.exp(logprob_drn_small_kl)
prob_drn_large_kl = np.exp(logprob_drn_large_kl)
prob_ddr = np.exp(logprob_ddr)
prob_baseline = np.exp(logprob_baseline)

# Add small epsilon to avoid division by zero or log(0)
epsilon = 1e-12
prob_drn = np.clip(prob_drn, epsilon, None)
prob_drn_small_kl = np.clip(prob_drn_small_kl, epsilon, None)
prob_drn_large_kl = np.clip(prob_drn_large_kl, epsilon, None)
prob_ddr = np.clip(prob_ddr, epsilon, None)

# Compute Δy (uniform grid spacing)
delta_y = (grid[1] - grid[0]).item()

# Compute KL divergence for each test point
kl_divs_drn = (
    np.sum(prob_drn * (np.log(prob_drn) - np.log(prob_baseline)), axis=1) * delta_y
)
kl_divs_drn_small_kl = (
    np.sum(
        prob_drn_small_kl * (np.log(prob_drn_small_kl) - np.log(prob_baseline)), axis=1
    )
    * delta_y
)
kl_divs_drn_large_kl = (
    np.sum(
        prob_drn_large_kl * (np.log(prob_drn_large_kl) - np.log(prob_baseline)), axis=1
    )
    * delta_y
)
kl_divs_ddr = (
    np.sum(prob_ddr * (np.log(prob_ddr) - np.log(prob_baseline)), axis=1) * delta_y
)
# Define consistent bins for overlapping histogram
bin_width = 0.01 / 2
max_val = max(np.max(kl_divs_drn), np.max(kl_divs_ddr))
bins = np.arange(0, max_val + bin_width, bin_width)

# Create the plot
plt.figure(figsize=(7, 7))

# Plot KDEs for each KL divergence group
sns.kdeplot(
    kl_divs_drn_large_kl,
    label="Model = DRN (KL_Reg = 0.05)",
    bw_adjust=1.5,
    linewidth=4,
    color="blue",
    alpha=0.25,
)
sns.kdeplot(
    kl_divs_drn,
    label="Model = DRN (KL_Reg = 0.003; Tuned)",
    bw_adjust=1.5,
    linewidth=4,
    color="blue",
    alpha=1.0,
)
sns.kdeplot(
    kl_divs_drn_small_kl,
    label="Model = DRN (KL_Reg = 0)",
    bw_adjust=1.5,
    linewidth=4,
    color="purple",
    alpha=1.0,
)
sns.kdeplot(kl_divs_ddr, label="Model = DDR", bw_adjust=1.5, linewidth=4, color="black")

# Axes and formatting
plt.title(
    "Distribution of $D_{\\text{KL}}(f_{\\text{Model}}||f_{\\text{GLM}})$ For Different Models",
    fontsize=16,
)
plt.xlabel("$D_{\\text{KL}}(f_{\\text{Model}}||f_{\\text{GLM}})$", fontsize=14)
plt.ylabel(
    "Empirical Density of $D_{\\text{KL}}(f_{\\text{Model}}||f_{\\text{GLM}})$",
    fontsize=14,
)
plt.xlim([0, 0.2])
# plt.ylim([0, 5000])
plt.legend(fontsize=14)
plt.tight_layout()
plt.savefig(PLOT_DIR / "_Synthetic_Distr_KL.png");